# VeriRAG Training Workflow

This notebook contains the training workflow for the VeriRAG verification model.

## 1. Project Introduction

**Objective:** Train a DeBERTa-v3 cross-encoder to verify claim veracity using retrieved context and RAGTruth-style document/claim pairs.
**Dataset:** RAGTruth benchmark or local dataset in `datasets/ragtruth_full.json`.
**Model:** `microsoft/deberta-v3-small` fine-tuned for 4-class classification.
**Labels:** SUPPORTED, PARTIALLY_SUPPORTED, CONTRADICTED, UNSUPPORTED.

## 2. Import Libraries

In [ ]:
import json
import time
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

from config import TrainConfig, id2label, label2id
from dataset import RAGTruthDataset, load_and_split_ragtruth

## 3. Configuration

In [ ]:
config = TrainConfig
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('====================================================')
print('Training Configuration')
print('====================================================')
print(f'Base Model:              {config.BASE_MODEL_NAME}')
print(f'Dataset:                 {config.DATASET_PATH}')
print(f'Epochs:                  {config.NUM_EPOCHS}')
print(f'Learning Rate:           {config.LEARNING_RATE}')
print(f'Batch Size:              {config.BATCH_SIZE}')
print(f'Gradient Accumulation:   {config.GRADIENT_ACCUMULATION_STEPS}')
print(f'Effective Batch Size:    {config.BATCH_SIZE * config.GRADIENT_ACCUMULATION_STEPS}')
print(f'Sequence Length:         {config.MAX_SEQ_LENGTH}')
print(f'Device:                  {device}')
print(f'Random Seed:             {config.SEED}')
print('====================================================')

## 4. Reproducibility

In [ ]:
import random

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(config.SEED)
print(f"Set seed = {config.SEED}")

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cuda_available = torch.cuda.is_available()
gpu_name = torch.cuda.get_device_name(0) if cuda_available else 'N/A'
cuda_version = torch.version.cuda if cuda_available else 'N/A'
gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3) if cuda_available else 0.0
print('====================================================')
print('Hardware Information')
print('====================================================')
print('Compute Device:', 'GPU' if cuda_available else 'CPU')
print('GPU Name:', gpu_name)
print('CUDA Version:', cuda_version)
print('PyTorch Version:', torch.__version__)
if cuda_available:
    print(f'Available GPU Memory: {gpu_memory_gb:.2f} GB')
print('====================================================')

## 5. Load RAGTruth Dataset

In [ ]:
train_samples, val_samples, test_samples = load_and_split_ragtruth(config.DATASET_PATH)
all_samples = train_samples + val_samples + test_samples
print(f"Total samples: {len(all_samples)}")
counts = Counter(sample['verdict'] for sample in all_samples)
print("Class distribution:")
for label, count in counts.items():
    print(f"- {label}: {count}")
print()
print(f"Train: {len(train_samples)} | Validation: {len(val_samples)} | Test: {len(test_samples)}")
missing_fields = [sample for sample in all_samples if any(field not in sample or sample[field] in [None, ''] for field in ['context', 'claim', 'verdict'])]
print(f"Missing or empty required fields: {len(missing_fields)}")

def lengths_for_samples(samples):
    return [len((sample['context'] + ' ' + sample['claim']).split()) for sample in samples]

all_lengths = lengths_for_samples(all_samples)
dataset_statistics = {
    'total_samples': len(all_samples),
    'train_samples': len(train_samples),
    'validation_samples': len(val_samples),
    'test_samples': len(test_samples),
    'class_distribution': dict(counts),
    'average_token_length': float(np.mean(all_lengths)),
    'maximum_token_length': int(np.max(all_lengths)),
    'minimum_token_length': int(np.min(all_lengths))
}
output_dir = Path(config.OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
with open(output_dir / 'dataset_statistics.json', 'w', encoding='utf-8') as f:
    json.dump(dataset_statistics, f, indent=2)
print(f"Saved dataset statistics to {output_dir / 'dataset_statistics.json'}")

## 6. Dataset Visualization

In [ ]:
def lengths_for_samples(samples):
    return [len((sample['context'] + ' ' + sample['claim']).split()) for sample in samples]

all_lengths = lengths_for_samples(all_samples)
label_freq = [counts[label] for label in ['SUPPORTED', 'PARTIALLY_SUPPORTED', 'CONTRADICTED', 'UNSUPPORTED']]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0, 0].bar(counts.keys(), counts.values(), color='tab:blue')
axes[0, 0].set_title('Class Distribution')
axes[0, 0].set_ylabel('Count')

axes[0, 1].bar(['SUPPORTED', 'PARTIALLY_SUPPORTED', 'CONTRADICTED', 'UNSUPPORTED'], label_freq, color='tab:orange')
axes[0, 1].set_title('Label Frequency')

axes[1, 0].hist(all_lengths, bins=20, color='tab:green', edgecolor='k')
axes[1, 0].set_title('Token Length Histogram')
axes[1, 0].set_xlabel('Token Count')

axes[1, 1].hist(all_lengths, bins=20, color='tab:red', edgecolor='k')
axes[1, 1].set_title('Average Sequence Length Distribution')
axes[1, 1].set_xlabel('Token Count')

plt.tight_layout()
plt.show()

## 7. Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(config.BASE_MODEL_NAME)
example = all_samples[0] if all_samples else {'context': '', 'claim': ''}
encoding = tokenizer(
    example['context'],
    example['claim'],
    truncation=True,
    max_length=config.MAX_SEQ_LENGTH,
    padding='max_length',
    return_tensors='pt'
)
print('Example text:')
print('Context:', example['context'])
print('Claim:', example['claim'])
print()
print('Tokenized input_ids shape:', encoding['input_ids'].shape)
print('First 20 token ids:', encoding['input_ids'][0, :20].tolist())
print('Decoded tokens:', tokenizer.convert_ids_to_tokens(encoding['input_ids'][0, :20]))

## 8. Dataset Preparation

In [ ]:
train_dataset = RAGTruthDataset(train_samples, tokenizer, max_length=config.MAX_SEQ_LENGTH)
val_dataset = RAGTruthDataset(val_samples, tokenizer, max_length=config.MAX_SEQ_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=config.EVAL_BATCH_SIZE, shuffle=False)

print('Train dataset size:', len(train_dataset))
print('Validation dataset size:', len(val_dataset))
batch = next(iter(train_loader))
print('Batch keys:', list(batch.keys()))
print('Input IDs shape:', batch['input_ids'].shape)
print('Attention mask shape:', batch['attention_mask'].shape)
print('Labels shape:', batch['labels'].shape)

## 9. Load Base Model

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = AutoModelForSequenceClassification.from_pretrained(
    config.BASE_MODEL_NAME,
    num_labels=config.NUM_CLASSES,
    id2label=id2label,
    label2id=label2id
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model loaded on {device}')
print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

## 10. Training Configuration

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)
num_update_steps_per_epoch = max(1, len(train_loader) // config.GRADIENT_ACCUMULATION_STEPS)
num_training_steps = num_update_steps_per_epoch * config.NUM_EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(num_training_steps * config.WARMUP_RATIO),
    num_training_steps=num_training_steps
)
criterion = nn.CrossEntropyLoss(weight=torch.tensor(config.CLASS_WEIGHTS, dtype=torch.float, device=device))

print('Optimizer: AdamW')
print('Scheduler: linear warmup')
print('Loss: CrossEntropyLoss with class weights')
print(f'Gradient accumulation steps: {config.GRADIENT_ACCUMULATION_STEPS}')
print(f'Early stopping patience: {config.EARLY_STOPPING_PATIENCE}')
print(f'Training steps: {num_training_steps}')

## 11. Training Loop

In [ ]:
def save_training_args(output_dir: Path):
    """Persist hyperparameters for reproducibility."""
    output_dir.mkdir(parents=True, exist_ok=True)
    args = {k: str(v) if isinstance(v, Path) else v for k, v in config.__dict__.items() if k.isupper()}
    with open(output_dir / 'training_args.json', 'w', encoding='utf-8') as f:
        json.dump(args, f, indent=2)


def compute_metrics(labels, preds):
    """Compute validation accuracy and macro-averaged classification metrics."""
    accuracy = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='macro', zero_division=0
    )
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
    }


history = {
    'train_loss': [],
    'validation_loss': [],
    'macro_precision': [],
    'macro_recall': [],
    'macro_f1': [],
    'validation_accuracy': [],
    'learning_rate': [],
    'epoch_time': [],
}
best_metric = float('-inf') if config.GREATER_IS_BETTER else float('inf')
num_bad_epochs = 0
best_model_path = Path(config.OUTPUT_DIR)
best_model_path.mkdir(parents=True, exist_ok=True)
training_start = time.time()
best_val_accuracy = 0.0
best_macro_f1 = 0.0

for epoch in range(1, config.NUM_EPOCHS + 1):
    epoch_start = time.time()
    model.train()
    train_loss = 0.0
    optimizer.zero_grad()

    for step, batch in enumerate(tqdm(train_loader, desc=f'Epoch {epoch} training')):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(outputs.logits, labels) / config.GRADIENT_ACCUMULATION_STEPS
        loss.backward()
        train_loss += loss.item() * config.GRADIENT_ACCUMULATION_STEPS

        if (step + 1) % config.GRADIENT_ACCUMULATION_STEPS == 0 or (step + 1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

    avg_train_loss = train_loss / len(train_loader)

    model.eval()
    val_losses = []
    all_targets = []
    all_preds = []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f'Epoch {epoch} validation'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs.logits, labels)
            val_losses.append(loss.item())
            preds = torch.argmax(outputs.logits, dim=1)
            all_targets.extend(labels.cpu().tolist())
            all_preds.extend(preds.cpu().tolist())

    avg_val_loss = sum(val_losses) / len(val_losses)
    metrics = compute_metrics(all_targets, all_preds)
    epoch_time = time.time() - epoch_start

    history['train_loss'].append(avg_train_loss)
    history['validation_loss'].append(avg_val_loss)
    history['macro_precision'].append(metrics['precision'])
    history['macro_recall'].append(metrics['recall'])
    history['macro_f1'].append(metrics['f1'])
    history['validation_accuracy'].append(metrics['accuracy'])
    history['learning_rate'].append(optimizer.param_groups[0]['lr'])
    history['epoch_time'].append(epoch_time)

    print(
        f'Epoch {epoch}/{config.NUM_EPOCHS} | Train Loss: {avg_train_loss:.4f} | '
        f'Val Loss: {avg_val_loss:.4f} | Val F1: {metrics["f1"]:.4f} | '
        f'Acc: {metrics["accuracy"]:.4f} | LR: {optimizer.param_groups[0]["lr"]:.6f} | '
        f'Time: {epoch_time:.1f}s'
    )

    if metrics['accuracy'] > best_val_accuracy:
        best_val_accuracy = metrics['accuracy']

    if metrics['f1'] > best_macro_f1:
        best_macro_f1 = metrics['f1']

    improved = metrics['f1'] > best_metric if config.GREATER_IS_BETTER else metrics['f1'] < best_metric
    if improved:
        best_metric = metrics['f1']
        num_bad_epochs = 0
        model.save_pretrained(best_model_path)
        tokenizer.save_pretrained(best_model_path)
        save_training_args(best_model_path)
        print(f'Saved best model checkpoint to {best_model_path}')
    else:
        num_bad_epochs += 1
        print(f'No improvement for {num_bad_epochs}/{config.EARLY_STOPPING_PATIENCE} epochs.')
        if num_bad_epochs >= config.EARLY_STOPPING_PATIENCE:
            print('Early stopping triggered.')
            break

training_time = time.time() - training_start
epochs_completed = len(history['train_loss'])

## 11.1 Training Summary and Exported Artifacts

In [ ]:
# Persist complete training history for reproducibility and paper reporting
with open(best_model_path / 'training_history.json', 'w', encoding='utf-8') as f:
    json.dump(history, f, indent=2)

experiment_summary = {
    'base_model': config.BASE_MODEL_NAME,
    'dataset': str(config.DATASET_PATH),
    'training_hyperparameters': {
        k: str(v) if isinstance(v, Path) else v
        for k, v in config.__dict__.items() if k.isupper()
    },
    'best_macro_f1': float(best_macro_f1),
    'best_validation_accuracy': float(best_val_accuracy),
    'training_time_seconds': float(training_time),
    'date': time.strftime('%Y-%m-%d %H:%M:%S'),
    'checkpoint_path': str(best_model_path),
}

with open(best_model_path / 'experiment_summary.json', 'w', encoding='utf-8') as f:
    json.dump(experiment_summary, f, indent=2)

print('====================================================')
print('Training Summary')
print('====================================================')
print(f'Best Macro F1:            {best_macro_f1:.4f}')
print(f'Best Validation Accuracy: {best_val_accuracy:.4f}')
print(f'Epochs Completed:         {epochs_completed}')
print(f'Training Time:            {training_time:.1f} s')
print(f'Checkpoint Location:      {best_model_path}')
print('====================================================')
print(f'Saved training history to {best_model_path / "training_history.json"}')
print(f'Saved experiment summary to {best_model_path / "experiment_summary.json"}')

## 12. Model Checkpoint

In [ ]:
print(f'Best model checkpoint directory: {best_model_path}')
print('Saved files:')
for item in sorted(best_model_path.iterdir()):
    print('-', item.name)

## 13. Training Curves

In [ ]:
plt.figure(figsize=(12, 8))
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['validation_loss'], label='Validation Loss')
plt.plot(history['macro_f1'], label='Validation Macro F1')
plt.xlabel('Epoch')
plt.ylabel('Value')
plt.title('Training and Validation Curves')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(best_model_path / 'training_curves.png', dpi=150)
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(history['learning_rate'], marker='o')
plt.xlabel('Epoch')
plt.ylabel('Learning Rate')
plt.title('Learning Rate Schedule')
plt.grid(True)
plt.tight_layout()
plt.savefig(best_model_path / 'learning_rate_schedule.png', dpi=150)
plt.show()